# Automation & Agentic AI — Practical Notebook (LangChain edition)
## STUDENT VERSION — fill in every `TODO`

**A hands-on companion to the session on automation and agentic AI, built with LangChain.**

---
### What you'll do in this notebook

- Connect to Google Gemini through LangChain.
- Look at a **worked example**: a 3-agent chain that turns a topic into a
  short report (Research → Summarize → Format).
- Complete the session's exercise: **Design a simple AI-assisted workflow**
  — then **build it** as your own chain of 2-3 agents.

### A few words explained up front

- **Chain** — a sequence of steps piped together with `|`. The output of one
  step becomes the input of the next.
- **Agent (in this notebook)** — a chain with a specific role (e.g.
  "Summarizer"), built from a prompt + an LLM (+ optionally a tool). One
  agent in the worked example also uses a real tool (web search) — that's
  the difference between a "chain" and a proper tool-using "agent" in the
  strict sense, and you'll see both.
- **LCEL (`|`)** — the pipe operator that connects steps: `prompt | llm | parser`
  reads left to right, like a small production line.

### How to use this notebook

Cells marked **`# TODO`** are yours to complete. Everything else is given —
read it, run it, and it should just work. Run cells **in order, top to
bottom** — later cells reuse the `llm` connection and helper functions set
up earlier.

> **How to run this:** Google Colab or a normal Jupyter notebook both work.
> No GPU needed. You'll need a free Gemini API key (instructions below).

## 1 · Install the libraries

Run this cell first.

In [1]:
!pip install -q langchain langchain-google-genai langgraph ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 824.4 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 6.5 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 32.5 MB/s eta 0:00:0000:0100:01


## 2 · Connect to the Gemini AI model

Get a free API key at **https://aistudio.google.com/apikey**

In [7]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL = "gemini-3.5-flash-lite"

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key here: ")

# This "llm" object is our connection to the AI model.
# Every agent/chain we build below will reuse this same connection.
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.4)

print("Connected! Using model:", MODEL)

Connected! Using model: gemini-3.5-flash-lite


## 3 · What does "chaining agents" mean in LangChain?

A single agent in this notebook is just three pieces glued together:

```
   a PROMPT (its instructions)  -->  the LLM  -->  an OUTPUT PARSER (cleans up the result)
```

Written in LangChain, that's literally: `prompt | llm | parser`. Read the `|`
as "then send the output of this into that" — the same idea as a Unix pipe.

**Chaining multiple agents** just means connecting several of these in a row,
where each one has its own role:

```
  Agent 1 (Research)  -->  Agent 2 (Analyze)  -->  Agent 3 (Write)
   raw information          prioritized facts +        flowing narrative
                             why each one matters       report, no bullets
```

This is a simple version of the "Orchestrator → Workers" pattern from the
session slides — except here the hand-off is a straight line instead of a
central orchestrator. Section 4 builds exactly this, with 3 agents.

## 4 · Worked example — a 3-agent chained workflow (given)

**The task:** turn a topic into a short, polished report.

- **Agent 1 — Researcher.** A real tool-using agent (it can call a web
  search tool) that gathers a few raw facts.
- **Agent 2 — Analyst.** A plain prompt-chain that doesn't just shorten the
  raw facts — it picks the most important ones and explains *why each one
  matters*, so there's real reasoning happening here, not just compression.
- **Agent 3 — Writer.** A plain prompt-chain that turns that analysis into a
  short **narrative report** — flowing sentences, not a list. This is a
  genuinely different kind of output than Agent 2 produces, which is the
  whole point of having a separate stage: each agent should transform its
  input into something meaningfully different, not just reformat it.

### 4.1 · Agent 1 — a tool-using Research agent

This one is a real agent in the strict sense: it can decide, on its own, to
call the `web_search` tool before answering. We build it with LangChain's
`create_agent` — the standard way to get a tool-calling agent without
writing the reasoning loop yourself (same idea as CrewAI's `Agent`, different
library).

In [3]:
!pip install ddgs

In [4]:
from langchain_core.tools import tool
from ddgs import DDGS

@tool
def web_search(query: str) -> str:
    """Search the web and return a few short results with titles and snippets."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return f"No results found for '{query}'."
        return "\n".join(f"- {r['title']}: {r['body'][:180]}" for r in results)
    except Exception as e:
        return f"Search failed for '{query}' ({e}). Try again in a moment."

# Sanity check -- call the tool directly, no agent involved yet.
print(web_search.invoke("population of Japan"))

- Population of Japan: The demography of Japan is monitored by National Institute of Population and Social Security Research (IPSS) and Statistics Bureau. In April 2025, Japan's population was roughly 12
- Demographics of Japan - Wikipedia: 1 day ago - The demography of Japan is monitored by National Institute of Population and Social Security Research (IPSS) and Statistics Bureau. In April 2025, Japan's population wa
- Japan - Wikipedia: 2 days ago - The Japanese archipelago consists ... islands. Japan is divided into 47 administrative prefectures and eight traditional regions, and around 75% of its terrain is moun


In [5]:
from langchain.agents import create_agent

research_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=(
        "You are a research agent. Use the web_search tool to gather a "
        "few raw facts about the topic you're given. Keep it brief and "
        "factual -- no formatting, no opinions, just the facts you found."
    ),
)

def run_research_agent(topic: str) -> str:
    """Run the research agent on a topic and return its final text answer."""
    result = research_agent.invoke({
        "messages": [{"role": "user", "content": f"Research this topic: {topic}"}]
    })
    return result["messages"][-1].content

# Sanity check
print(run_research_agent("the current price of Bitcoin in USD"))

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The current price of Bitcoin (BTC) in USD is approximately $77,200 to $77,240, though cryptocurrency prices fluctuate constantly in real-time.', 'extras': {'signature': 'El4KXAERTTIPH+O8p5ix+OwxB05IZIhZTmFfi/qy1F4Asdu8fNJjir9P39FtpSQ34GoTGwpXxLCqdq5+r7OGptE6BlHHFy0HFhrNuuO8GHlEHef8EyjBTQ1tbq0WFJJC'}}]


### 4.2 · Agent 2 — an Analyst (plain chain, no tools)

Not every agent needs a tool. This one just needs a clear role and a focused
prompt. Notice the shape: `prompt | llm | parser` -- that's the whole agent.

Its job is deliberately **more than shortening text**: it has to *decide*
which facts matter most and *explain why* -- that's a small piece of
reasoning, not just compression. That's what makes it worth being its own
agent instead of a copy-paste step.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summarizer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an analysis agent. You'll be given raw, unstructured research "
     "notes. Do three things:\n"
     "1. Pick the 3 most important facts -- drop anything trivial or repeated.\n"
     "2. For each fact, add a short 'why it matters' explanation in the same line.\n"
     "3. If anything in the notes looks uncertain, outdated, or contradictory, "
     "flag it in one line at the end starting with 'Note:'.\n"
     "Output as a numbered list: `1. <fact> -- <why it matters>`. This is "
     "intermediate output for another agent, not the final report, so skip "
     "any greetings or closing remarks."),
    ("human", "{raw_notes}"),
])

summarizer_chain = summarizer_prompt | llm | StrOutputParser()

# Sanity check -- test with made-up raw notes, no research agent involved.
# Notice the output isn't just shorter -- it explains why each fact matters.
print(summarizer_chain.invoke({"raw_notes": "Cats sleep 12-16 hours a day. "
                                             "Cats have 32 muscles in each ear. "
                                             "A group of cats is called a clowder."}))

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


1. Cats sleep 12-16 hours a day -- it highlights their significant need for rest and energy conservation.
2. Cats have 32 muscles in each ear -- it explains their exceptional hearing capabilities and ability to rotate their ears 180 degrees.
3. A group of cats is called a clowder -- it provides specific terminology used to describe feline social groupings.

Note: The notes do not contain any obviously uncertain, outdated, or contradictory information.


### 4.3 · Agent 3 — a Writer (plain chain, no tools)

Same shape again -- a different role and prompt -- but this time the job is
to genuinely **rewrite**, not just restyle. Agent 2 hands over a numbered,
analytical list; Agent 3's job is to turn that into flowing prose a person
would actually enjoy reading. If this agent's output looked just like Agent
2's with a sentence tacked on, it wouldn't be earning its place in the
chain -- so its prompt explicitly forbids lists and bullets.

In [8]:
formatter_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a writing agent. You'll be given a short numbered analysis "
     "(facts plus why each one matters). Rewrite it as a short narrative "
     "report for a general reader: 3-4 flowing, connected sentences. "
     "Open with a one-line hook, weave the facts and why they matter "
     "naturally into the prose, and end with one forward-looking or "
     "takeaway sentence. Do NOT use a list, bullet points, numbers, or "
     "headers -- plain paragraph prose only."),
    ("human", "{summary}"),
])

formatter_chain = formatter_prompt | llm | StrOutputParser()

# Sanity check -- feed it Agent 2's *style* of output (a numbered analysis)
# and confirm you get back prose, not the same list restated.
print(formatter_chain.invoke({"summary":
    "1. Cats sleep 12-16 hours a day -- indoor cats need engaging play during "
    "their shorter waking hours.\n"
    "2. Cats have 32 muscles in each ear -- this gives them excellent "
    "directional hearing for hunting.\n"
    "3. A group of cats is called a 'clowder' -- a fun fact people rarely know."
}))

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Did you know that a group of cats is officially called a clowder? While these mysterious companions spend twelve to sixteen hours a day sleeping, their waking hours are packed with incredible biological superpowers, like the thirty-two muscles in each ear that grant them pinpoint directional hearing once honed for hunting. Because indoor cats miss out on chasing real prey, they rely heavily on engaging playtime with their humans to stay mentally and physically sharp during their brief windows of alertness. Understanding these fascinating traits helps us become better pet parents, ensuring our furry friends live happy, enriched lives both awake and asleep.


### 4.4 · Run the whole workflow, step by step

Now chain all three agents by hand: run Agent 1, feed its output into
Agent 2, feed *that* output into Agent 3.

In [9]:
topic = "agentic AI being used in classrooms"

raw_notes = run_research_agent(topic)
print("=== AGENT 1 (Research) ===")
print(raw_notes)

summary = summarizer_chain.invoke({"raw_notes": raw_notes})
print("\n=== AGENT 2 (Summarize) ===")
print(summary)

final_report = formatter_chain.invoke({"summary": summary})
print("\n=== AGENT 3 (Format) -- FINAL ANSWER ===")
print(final_report)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== AGENT 1 (Research) ===
[{'type': 'text', 'text': "- **Definition of Agentic AI:** Unlike traditional generative AI tools (which primarily respond reactively to prompts), agentic AI systems possess autonomy, goal-directed behavior, and the ability to make decisions and execute multi-step workflows with minimal human intervention.\n- **Personalized Tutoring & Student Support:** AI agents act as 24/7 personal tutors that can adapt to a student's unique learning pace, identify knowledge gaps, adjust difficulty levels dynamically, and guide learners through complex problem-solving without just giving away answers.\n- **Teacher Augmentation & Administrative Relief:** AI agents help educators by automating repetitive, time-consuming tasks such as grading formative assessments, drafting lesson plans, tracking student engagement, and managing administrative workloads, thereby freeing up time for direct student interaction.\n- **Curriculum and Course Generation:** Autonomous agents assist in

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



=== AGENT 2 (Summarize) ===
1. Agentic AI features autonomy, goal-directed behavior, and multi-step execution -- unlike reactive generative tools, it can independently manage complex workflows with minimal human oversight.
2. AI agents provide 24/7 personalized tutoring and student support -- they dynamically adapt to learning paces and identify knowledge gaps without simply giving away answers.
3. Teacher augmentation and administrative relief are core benefits -- automating grading, lesson planning, and paperwork frees up educators for direct student interaction.

Note: The notes outline broad capabilities and standard risks of agentic AI in education, but lack specific empirical data or implementation timelines to verify the current maturity of these systems.


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



=== AGENT 3 (Format) -- FINAL ANSWER ===
Agentic AI is moving classrooms beyond simple reactive tools by introducing autonomous systems capable of independently managing complex, multi-step workflows. By providing round-the-clock personalized tutoring that dynamically adapts to individual learning paces and pinpoints knowledge gaps without spoiling answers, these systems foster deeper student engagement. Furthermore, they offer vital administrative relief by automating grading, lesson planning, and paperwork, which ultimately frees educators to focus on meaningful, direct student interaction. As these technologies continue to evolve, successfully bridging the gap between broad capabilities and proven, real-world classroom implementation will be key to shaping the future of education.


### Bonus (given) — the same workflow as one LCEL chain

Everything above can be written as a **single piped expression** — this is
the "chain" in LangChain. `RunnableLambda` just wraps a plain function so it
can sit inside a `|` pipeline alongside the prompt-chains.

In [10]:
from langchain_core.runnables import RunnableLambda

full_workflow = (
    RunnableLambda(lambda topic: run_research_agent(topic))
    | RunnableLambda(lambda raw_notes: {"raw_notes": raw_notes})
    | summarizer_chain
    | RunnableLambda(lambda summary: {"summary": summary})
    | formatter_chain
)

result = full_workflow.invoke("agentic AI being used in classrooms")
print(result)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

Educational technology is quietly evolving from passive chatbots into autonomous agents capable of independent decision-making and multi-step execution. By dynamically monitoring student progress and adjusting curricula on the fly, these intelligent systems offer a scalable solution to the traditional limits of one-on-one human tutoring. Beyond the classroom, they act as powerful co-pilots for instructors by automating grading, lesson planning, and even dropout prediction to drastically reduce faculty workloads. However, as these tools edge closer to total autonomy, establishing strict guardrails against data privacy risks and algorithmic bias will remain essential to ensure they enhance learning safely.


Same result, one expression. Both versions are "correct" LangChain — the
step-by-step version is easier to debug (you can print each stage), the
piped version is more idiomatic once you trust each piece. Use whichever
you prefer for your own workflow below.

## 5 · Exercise — Design a simple AI-assisted workflow

Pick **one small task** you actually do, that naturally breaks into 2-3
stages where each stage has a clearly different job. That's what makes it a
good fit for chaining. Some ideas:

- Raw meeting notes → extracted action items → a clean checklist message
- A list of messy data → cleaned-up data → a short written summary
- A question → researched facts → a beginner-friendly explanation

Fill in the blanks below. This is planning only — no agents built yet.

In [12]:
# TODO 5.1 -- Fill in every blank with your own answers.

task_name = "Messy notes to a clean study summary"                  # e.g. "Meeting notes to action items"

# List your agents IN ORDER. Each one should have a narrow, clear job --
# that's what makes it easy to write a good prompt for it.
# TODO 5.1 -- Fill in every blank with your own answers.

task_name = "Messy notes to a clean study summary"

agent_plan = [
    {"agent_name": "Extractor", "job": "extract the important points from the raw notes"},
    {"agent_name": "Summarizer", "job": "turn the extracted points into a short and clear study summary"},
    {"agent_name": "Formatter", "job": "organize the summary into headings and bullet points"}
]

does_any_agent_need_a_tool = "no"

one_guardrail = "Do not add information that is not present in the original notes"

print("Task:", task_name)
for a in agent_plan:
    print(" -", a["agent_name"], "->", a["job"])
print("Needs a tool?", does_any_agent_need_a_tool)
print("Guardrail:", one_guardrail)

does_any_agent_need_a_tool = "no"   # "yes" or "no" -- and if yes, which one?

one_guardrail = "Do not add information that is not present in the original notes"                # TODO: one safety limit you'd add,
                                      # e.g. "always show the draft before sending"

print("Task:", task_name)
for a in agent_plan:
    print(" -", a["agent_name"], "->", a["job"])
print("Needs a tool?", does_any_agent_need_a_tool)
print("Guardrail:", one_guardrail)

Task: Messy notes to a clean study summary
 - Extractor -> extract the important points from the raw notes
 - Summarizer -> turn the extracted points into a short and clear study summary
 - Formatter -> organize the summary into headings and bullet points
Needs a tool? no
Guardrail: Do not add information that is not present in the original notes
Task: Messy notes to a clean study summary
 - Extractor -> extract the important points from the raw notes
 - Summarizer -> turn the extracted points into a short and clear study summary
 - Formatter -> organize the summary into headings and bullet points
Needs a tool? no
Guardrail: Do not add information that is not present in the original notes


## 6 · Build your chained-agent workflow

Build **at least 2 agents** from your `agent_plan` above, following the exact
pattern from Section 4: a `ChatPromptTemplate`, piped into `llm`, piped into
`StrOutputParser()`. A tool is optional — copy the Section 4.1 pattern only
if your design actually needs one.

#### TODO 6.1 — Build Agent 1 from your plan

In [14]:
# TODO 6.1 -- Build your first agent. Copy the shape of `summarizer_chain`
# from Section 4.2 (or `research_agent` from 4.1, if this stage needs a tool).

agent1_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Extractor. Extract only the important points from the raw notes. Do not add any information that is not present in the notes."),   # TODO: describe this agent's one job, clearly
    ("human", "{raw_notes}"),  # TODO: name the input variable this agent expects
])

agent1_chain = agent1_prompt | llm | StrOutputParser()

# Sanity check -- test it on its own with made-up input before chaining anything.
print(agent1_chain.invoke({"raw_notes": "Python is a popular programming language. It is easy to learn and is widely used in AI, data science, and web development."}))   # TODO: match the key you used above

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


- Python is a popular programming language.
- It is easy to learn.
- It is widely used in AI, data science, and web development.


#### TODO 6.2 — Build Agent 2 from your plan

This one should take Agent 1's *output* as its input.

In [15]:
# TODO 6.2 -- Build your second agent, same pattern as 6.1.

agent2_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Summarizer. Turn the extracted important points into a short, clear, and easy-to-study summary. Do not add new information"),   # TODO
    ("human", "{extracted_points}"),  # TODO
])

agent2_chain = agent2_prompt | llm | StrOutputParser()

# Sanity check
print(agent2_chain.invoke({"extracted_points": "Python is a popular programming language. It is easy to learn.  It is widely used in AI, data science, and web development."}))   # TODO

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Python Summary:**

* **Popularity:** A widely used programming language.
* **Usability:** Easy to learn.
* **Applications:** Used in AI, data science, and web development.


#### (Optional) TODO 6.3 — Build a third agent

Only if your `agent_plan` has 3 stages. Skip this cell if 2 agents cover
your task.

In [16]:
# TODO 6.3 (optional) -- Build a third agent, same pattern, if your plan needs one.

agent3_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Formatter. Organize the study summary into clear headings and bullet points. Do not add new information."),   # TODO
    ("human", "{summary}"),  # TODO
])

agent3_chain = agent3_prompt | llm | StrOutputParser()

print(agent3_chain.invoke({"summary": "Python is easy to learn and is widely used in AI, data science, and web development."}))   # TODO

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Study Summary: Python**

* **Ease of Learning**
  * Python is easy to learn.

* **Primary Applications**
  * Artificial Intelligence (AI)
  * Data Science
  * Web Development


#### TODO 6.4 — Run your full chain, step by step

Copy the pattern from Section 4.4: run agent 1, feed its output into agent 2
(and agent 3, if you built one), print each stage.

In [17]:
# TODO 6.4 -- Run your agents in order, passing each output to the next input.

my_input = """"Python is a popular programming language.
It is easy to learn.
Python is widely used in artificial intelligence, data science, and web development.
It has many libraries that help developers build applications quickly."""   # TODO: a real example input for your task

step1_output = agent1_chain.invoke({"raw_notes": my_input})   # TODO: match your key
print("=== AGENT 1 ===")
print(step1_output)

step2_output = agent2_chain.invoke({"extracted_points": step1_output})   # TODO: match your key
print("\n=== AGENT 2 -- FINAL ANSWER (or pass to agent 3) ===")
print(step2_output)

# TODO (optional): if you built agent3_chain, add the same pattern here.

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== AGENT 1 ===
* Python is a popular, easy-to-learn programming language.
* It is widely used in artificial intelligence, data science, and web development.
* It has many libraries that help developers build applications quickly.


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



=== AGENT 2 -- FINAL ANSWER (or pass to agent 3) ===
**Python Summary:**

* **Overview:** A popular and easy-to-learn programming language.
* **Applications:** Widely used in artificial intelligence, data science, and web development.
* **Efficiency:** Features many libraries that enable developers to build applications quickly.


## 7 · Wrap-up (written answer)

Edit this cell:

1. Did splitting the task across multiple agents actually help, or would
   one agent with a longer prompt have done just as well?
2. Would any stage benefit from a real tool (like the Research agent's web
   search)? Which one, and why?
3. Looking back at your `one_guardrail` from Section 5 — where exactly would
   you add it in the chain you just built?
# Answers
1. Yes, splitting the task across multiple agents helped because each agent had one clear responsibility. The Extractor selected the important information, the Summarizer simplified it, and the Formatter organized the final result. This made the workflow easier to understand and control
2. A tool could be useful if one stage needed external information, such as searching the web or reading a file. In this workflow, no tool is necessary because all agents work only with the provided notes.
3. I would add the guardrail to every agent, especially the Extractor and Summarizer, to make sure they do not add information that is not present in the original notes.